# Week 13: Reinforcement Learning & RLHF
## Notebook 08: Multi-Agent RL: CTDE & QMIX Value Factorization

### 1. Conceptual Intuition & Theoretical Background
In Multi-Agent Reinforcement Learning (MARL), independent learning (e.g. Independent Q-Learning, IQL) fails because the environment appears **non-stationary** from the perspective of any single agent as other agents update their policies simultaneously.

**Centralized Training with Decentralized Execution (CTDE)** resolves this:
- **During Training:** A centralized critic/mixing network observes global state $s_t$ and all agent actions.
- **During Execution:** Each agent $a$ selects actions decentralised using only its local observation history $o^a$.

**QMIX** factorizes the joint action-value function $Q_{\text{tot}}(\mathbf{s}, \mathbf{u})$ while preserving the **Individual-Global-Max (IGM)** condition:
$$\arg\max_{\mathbf{u}} Q_{\text{tot}}(\mathbf{s}, \mathbf{u}) = \begin{pmatrix} \arg\max_{u_1} Q_1(s, u_1) \\ \dots \\ \arg\max_{u_N} Q_N(s, u_N) \end{pmatrix}$$

QMIX enforces this by constraining the mixing network weights to be non-negative:
$$\frac{\partial Q_{\text{tot}}}{\partial Q_a} \ge 0, \quad \forall a$$
Using hypernetworks that produce absolute weights:
$$W = |\text{HyperNet}(s)|$$

---

### 2. Scratch Python Implementation: QMIX Mixing Network

In [ ]:
import numpy as np

class QMIXMixingNetworkScratch:
    def __init__(self, n_agents=3, state_dim=12, mixing_embed_dim=16):
        self.n_agents = n_agents
        self.state_dim = state_dim
        self.embed_dim = mixing_embed_dim
        
        # Hypernetwork 1: Maps global state s -> Absolute Weights W1 of shape (n_agents, embed_dim)
        np.random.seed(42)
        self.hw1 = np.random.normal(0, 0.1, (state_dim, n_agents * mixing_embed_dim))
        self.hb1 = np.zeros(mixing_embed_dim)
        
        # Hypernetwork 2: Maps global state s -> Absolute Weights W2 of shape (embed_dim, 1)
        self.hw2 = np.random.normal(0, 0.1, (state_dim, mixing_embed_dim))
        self.hb2 = np.zeros(1)
        
    def forward(self, agent_qs, global_state):
        """
        agent_qs: Individual utilities shape (batch_size, n_agents)
        global_state: True global state shape (batch_size, state_dim)
        """
        batch_size = len(agent_qs)
        
        # Generate W1 from state and enforce absolute value for monotonicity
        w1_raw = np.dot(global_state, self.hw1).reshape(batch_size, self.n_agents, self.embed_dim)
        W1 = np.abs(w1_raw)  # Strict non-negativity constraint!
        
        # First layer mixing
        # (B, 1, N) @ (B, N, Embed) -> (B, 1, Embed)
        q_reshaped = agent_qs[:, np.newaxis, :]
        hidden = np.matmul(q_reshaped, W1) + self.hb1
        hidden = np.maximum(0, hidden) # ELU / ReLU
        
        # Generate W2 from state
        w2_raw = np.dot(global_state, self.hw2).reshape(batch_size, self.embed_dim, 1)
        W2 = np.abs(w2_raw)
        
        # Output Q_tot
        q_tot = np.matmul(hidden, W2) + self.hb2
        return q_tot.squeeze()

# Simulate 3 Cooperative Robotic Agents
n_agents = 3
state_dim = 8
batch_size = 4

# Individual utilities computed by independent agent networks
agent_utilities = np.array([
    [1.5, 2.0, 0.5],
    [3.0, 1.0, 4.0],
    [-0.5, 0.2, 1.1],
    [2.2, 2.5, 2.8]
])

global_states = np.random.randn(batch_size, state_dim)

qmix = QMIXMixingNetworkScratch(n_agents=3, state_dim=8)
q_total_values = qmix.forward(agent_utilities, global_states)

print(f"Batch Size:                       {batch_size}")
print(f"Decentralized Agent Q-Values [0]: {agent_utilities[0]}")
print(f"Joint Q_tot Estimated Value [0]:  {q_total_values[0]:.4f}")
print("Monotonic factorization guarantees optimal decentralized action selection!")

### 3. Step-by-Step Code Walkthrough
1. **Hypernetwork Conditioning**: Mixing weights are dynamically synthesized from global state $s$.
2. **Absolute Value Enforcement**: `np.abs(W)` guarantees $\frac{\partial Q_{\text{tot}}}{\partial Q_a} \ge 0$.
3. **Decentralized Execution**: An individual agent needs only its local $Q_a$ to pick its greedy action.

---

### 4. Industry Applications & Trade-offs
- **Fleet Logistics & Warehouse Robotics:** Coordinating autonomous mobile robots without central communication bottlenecks.
- **Trade-off:** Monotonicity constraint prevents modeling tasks requiring sub-optimal individual actions to achieve a global optimum (e.g. predator-prey traps).

---

### 5. Top Interview Questions & Answers
**Q1: What is the Individual-Global-Max (IGM) condition in cooperative MARL?**
> **Answer:** It asserts that the joint action that maximizes the global centralized $Q_{\text{tot}}$ must be identical to the collection of individual greedy actions chosen by each decentralized agent.

**Q2: Why does QMIX enforce positive weights in its mixing network?**
> **Answer:** By constraining weights to be strictly positive (via absolute values or exponential functions), any increase in an individual agent's utility $Q_a$ monotonically increases $Q_{\text{tot}}$, mathematically preserving the IGM condition.